# **Homework 2 - Image Captioning**

<div style="border: 3px solid #222; padding: 16px; border-radius: 10px; background-color: #1c1f26; font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif; color: #e0e0e0;">
  <div style="display: flex; align-items: center; gap: 8px; margin-top: 12px;">
    <span style="font-size: 24px; color: #ff5555;">&#128274;</span>
    <span style="font-size: 16px;"><strong>Project:</strong> Homeworks</span>
  </div>
  <div style="display: flex; align-items: center; gap: 8px; margin-top: 12px;">
    <span style="font-size: 24px; color: #ff5555;">&#128218;</span>
    <span style="font-size: 16px;"><strong>Course:</strong> Deep Network Development 25/26/2</span>
  </div>
  <div style="display: flex; align-items: center; gap: 8px; margin-top: 12px;">
    <span style="font-size: 24px; color: #6e8192;">&#128100;</span>
    <span style="font-size: 16px;"><strong>Authors:</strong> Tamás Takács (PhD student, Department of Artificial Intelligence, Eötvös Loránd University) </span>
  </div>
</div>
<hr style="border: none; border-top: 2px solid #444;">
<br>


**Name:**  
**Neptun ID:**

## **0. Necessary Imports**


In [ ]:
import os
import json
import math
import copy
import time
import heapq
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.models as models
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import random
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from collections import Counter
from tqdm import tqdm

# NLTK for BLEU
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction

# pycocotools
# !pip install pycocotools
from pycocotools.coco import COCO

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

## **1. Data Loading Process**


In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────
DATA_ROOT      = './coco'
TRAIN_IMG_DIR  = os.path.join(DATA_ROOT, 'images', 'train2017')
VAL_IMG_DIR    = os.path.join(DATA_ROOT, 'images', 'val2017')
TRAIN_ANN_FILE = os.path.join(DATA_ROOT, 'annotations', 'captions_train2017.json')
VAL_ANN_FILE   = os.path.join(DATA_ROOT, 'annotations', 'captions_val2017.json')

# ── Optional subset flag (set to None to use full dataset) ─────────────
TRAIN_SUBSET = 20000   # use ~20k images to fit modest hardware; set None for full
VAL_SUBSET   = 2000

# ── Download helper (uncomment if needed) ──────────────────────────────
# import subprocess
# os.makedirs(TRAIN_IMG_DIR, exist_ok=True)
# os.makedirs(VAL_IMG_DIR, exist_ok=True)
# os.makedirs(os.path.join(DATA_ROOT,'annotations'), exist_ok=True)
# subprocess.run(['wget','-q','http://images.cocodataset.org/zips/train2017.zip','-P', DATA_ROOT])
# subprocess.run(['wget','-q','http://images.cocodataset.org/zips/val2017.zip',  '-P', DATA_ROOT])
# subprocess.run(['wget','-q','http://images.cocodataset.org/annotations/annotations_trainval2017.zip','-P', DATA_ROOT])
# subprocess.run(['unzip','-q', os.path.join(DATA_ROOT,'train2017.zip'),          '-d', os.path.join(DATA_ROOT,'images')])
# subprocess.run(['unzip','-q', os.path.join(DATA_ROOT,'val2017.zip'),            '-d', os.path.join(DATA_ROOT,'images')])
# subprocess.run(['unzip','-q', os.path.join(DATA_ROOT,'annotations_trainval2017.zip'), '-d', DATA_ROOT])

print('Data paths configured.')
print(f'  Train images : {TRAIN_IMG_DIR}')
print(f'  Val images   : {VAL_IMG_DIR}')
print(f'  Train subset : {TRAIN_SUBSET}')
print(f'  Val subset   : {VAL_SUBSET}')

## **2. Defining Augmentations**


In [ ]:
# ImageNet stats
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

print('Transforms defined.')

## **3. Creating Datasets and Dataloaders**


In [ ]:
# ── Vocabulary ─────────────────────────────────────────────────────────
class Vocabulary:
    """Maps words <-> indices; only words appearing >= freq_threshold times are kept."""

    PAD, SOS, EOS, UNK = '<pad>', '<sos>', '<eos>', '<unk>'

    def __init__(self, freq_threshold: int = 5):
        self.freq_threshold = freq_threshold
        self.word2idx = {self.PAD: 0, self.SOS: 1, self.EOS: 2, self.UNK: 3}
        self.idx2word = {v: k for k, v in self.word2idx.items()}

    def __len__(self):
        return len(self.word2idx)

    @staticmethod
    def tokenize(text: str):
        return text.lower().split()

    def build_vocabulary(self, captions):
        freq = Counter()
        for cap in captions:
            freq.update(self.tokenize(cap))
        for word, cnt in freq.items():
            if cnt >= self.freq_threshold:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx]  = word
        print(f'Vocabulary built: {len(self)} tokens')

    def numericalize(self, text: str):
        unk = self.word2idx[self.UNK]
        return [self.word2idx.get(w, unk) for w in self.tokenize(text)]

    def decode(self, indices):
        tokens = []
        for i in indices:
            w = self.idx2word.get(int(i), self.UNK)
            if w == self.EOS:
                break
            if w not in (self.PAD, self.SOS):
                tokens.append(w)
        return ' '.join(tokens)


# ── Dataset ────────────────────────────────────────────────────────────
class COCOCaptionsDataset(Dataset):
    def __init__(self, img_dir, ann_file, vocab, transform=None, subset=None):
        self.img_dir   = img_dir
        self.vocab     = vocab
        self.transform = transform

        coco = COCO(ann_file)
        img_ids = list(coco.imgs.keys())
        if subset:
            img_ids = img_ids[:subset]

        # Collect (file_name, caption) pairs; store one caption per sample
        self.samples = []
        # Also keep ALL 5 refs per image for BLEU evaluation
        self.img_refs = {}  # img_id -> [list of tokenized captions]

        for img_id in img_ids:
            info     = coco.imgs[img_id]
            ann_ids  = coco.getAnnIds(imgIds=img_id)
            anns     = coco.loadAnns(ann_ids)
            captions = [a['caption'] for a in anns]
            file_name = info['file_name']
            for cap in captions:
                self.samples.append((file_name, cap))
            self.img_refs[img_id] = [vocab.tokenize(c) for c in captions]

        print(f'Dataset size: {len(self.samples)} caption samples')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        file_name, caption = self.samples[idx]
        img_path = os.path.join(self.img_dir, file_name)
        image    = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        numericalized = ([self.vocab.word2idx[self.vocab.SOS]]
                         + self.vocab.numericalize(caption)
                         + [self.vocab.word2idx[self.vocab.EOS]])
        caption_tensor = torch.tensor(numericalized, dtype=torch.long)
        return image, caption_tensor, len(caption_tensor)


# ── Collate ────────────────────────────────────────────────────────────
def collate_fn(batch):
    images, captions, lengths = zip(*batch)
    images  = torch.stack(images, 0)
    max_len = max(lengths)
    pad_idx = 0  # <pad>
    padded  = torch.full((len(captions), max_len), pad_idx, dtype=torch.long)
    for i, cap in enumerate(captions):
        padded[i, :len(cap)] = cap
    # Attention mask: True = ignore (padding positions)
    attn_mask = (padded == pad_idx)
    return images, padded, torch.tensor(lengths), attn_mask


# ── Build vocab on train captions ──────────────────────────────────────
print('Loading train annotations to build vocabulary...')
coco_tmp = COCO(TRAIN_ANN_FILE)
all_train_captions = [a['caption'] for a in coco_tmp.anns.values()]
vocab = Vocabulary(freq_threshold=5)
vocab.build_vocabulary(all_train_captions)

# ── Datasets ───────────────────────────────────────────────────────────
train_dataset = COCOCaptionsDataset(TRAIN_IMG_DIR, TRAIN_ANN_FILE, vocab,
                                     transform=train_transforms, subset=TRAIN_SUBSET)
val_dataset   = COCOCaptionsDataset(VAL_IMG_DIR,   VAL_ANN_FILE,   vocab,
                                     transform=val_transforms,   subset=VAL_SUBSET)

# ── DataLoaders ────────────────────────────────────────────────────────
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2, pin_memory=True, collate_fn=collate_fn)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## **4. Visualizing Training Data**


In [ ]:
def denormalize(tensor):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor(MEAN).view(3,1,1)
    std  = torch.tensor(STD).view(3,1,1)
    return (tensor * std + mean).clamp(0,1)


def visualize_batch(dataloader, vocab, num_samples=8):
    images, captions, lengths, _ = next(iter(dataloader))
    fig, axes = plt.subplots(2, num_samples//2, figsize=(20, 8))
    axes = axes.flatten()
    for i in range(num_samples):
        img  = denormalize(images[i]).permute(1,2,0).numpy()
        cap_tokens = captions[i][:lengths[i]].tolist()
        original   = vocab.decode(cap_tokens)
        token_ids  = str(cap_tokens[:10]) + ('...' if lengths[i]>10 else '')
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(f'{original}\n{token_ids}', fontsize=7, wrap=True)
    plt.suptitle('Sample training images with original and tokenized captions', fontsize=12)
    plt.tight_layout()
    plt.show()


visualize_batch(train_loader, vocab, num_samples=8)

## **5. Creating the Image Captioning Model**


In [ ]:
# ── Encoder ────────────────────────────────────────────────────────────
class CNNEncoder(nn.Module):
    """
    ResNet-50 backbone (fine-tuned from layer3 onwards).
    Outputs (B, num_patches, encoder_dim) spatial feature vectors.
    """
    def __init__(self, encoded_image_size=14, fine_tune_from='layer3'):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # Remove avgpool and fc
        modules = list(resnet.children())[:-2]
        self.resnet = nn.Sequential(*modules)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((encoded_image_size, encoded_image_size))
        self.encoder_dim = 2048

        # Freeze early layers
        layers = ['layer1','layer2','layer3','layer4']
        freeze_until = layers.index(fine_tune_from)
        for name, param in resnet.named_parameters():
            layer_name = name.split('.')[0]
            if layer_name not in layers[freeze_until:]:
                param.requires_grad = False

    def forward(self, images):
        # (B,2048,H,W)
        feats = self.resnet(images)
        feats = self.adaptive_pool(feats)            # (B,2048,14,14)
        B, C, H, W = feats.shape
        feats = feats.permute(0,2,3,1)               # (B,14,14,2048)
        feats = feats.view(B, H*W, C)                # (B,196,2048)
        return feats


# ── Positional Encoding ────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        pe = pe.unsqueeze(0)   # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


# ── Transformer Decoder Layer ──────────────────────────────────────────
class TransformerDecoderLayer(nn.Module):
    """
    Single Transformer decoder layer:
      1. Masked multi-head self-attention
      2. Cross-attention to encoder features  (returns attention weights)
      3. Feed-forward network
    """
    def __init__(self, d_model, num_heads, dim_feedforward, dropout=0.1):
        super().__init__()
        self.self_attn  = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.norm3   = nn.LayerNorm(d_model)
        self.drop1   = nn.Dropout(dropout)
        self.drop2   = nn.Dropout(dropout)
        self.drop3   = nn.Dropout(dropout)

    def forward(self, tgt, memory, tgt_mask=None, tgt_key_padding_mask=None):
        # 1. Masked self-attention
        sa_out, _ = self.self_attn(tgt, tgt, tgt,
                                    attn_mask=tgt_mask,
                                    key_padding_mask=tgt_key_padding_mask)
        tgt = self.norm1(tgt + self.drop1(sa_out))
        # 2. Cross-attention
        ca_out, attn_weights = self.cross_attn(tgt, memory, memory)
        tgt = self.norm2(tgt + self.drop2(ca_out))
        # 3. FFN
        tgt = self.norm3(tgt + self.drop3(self.ffn(tgt)))
        return tgt, attn_weights   # attn_weights: (B, tgt_len, src_len)


# ── Transformer Decoder ────────────────────────────────────────────────
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers,
                 dim_feedforward, encoder_dim, dropout=0.1, num_controls=4):
        super().__init__()
        self.d_model = d_model
        self.embedding      = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoding   = PositionalEncoding(d_model, dropout=dropout)
        # Project encoder features to d_model
        self.encoder_proj   = nn.Linear(encoder_dim, d_model)
        # Controllable generation: length-bucket embedding
        # Buckets: 0=short(<10), 1=medium(10-20), 2=long(20-35), 3=very_long(>35)
        self.control_embed  = nn.Embedding(num_controls, d_model)
        self.layers         = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, dim_feedforward, dropout)
            for _ in range(num_layers)
        ])
        self.output_proj    = nn.Linear(d_model, vocab_size)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.output_proj.weight)
        nn.init.zeros_(self.output_proj.bias)

    def generate_square_subsequent_mask(self, sz):
        """Causal mask: upper triangle filled with -inf."""
        mask = torch.triu(torch.ones(sz, sz), diagonal=1).bool()
        return mask.to(next(self.parameters()).device)

    def forward(self, captions, encoder_features, tgt_mask=None,
                tgt_key_padding_mask=None, control_signal=None):
        """
        captions:          (B, T)
        encoder_features:  (B, num_patches, encoder_dim)
        control_signal:    (B,) int tensor with bucket index [0-3], or None
        Returns: logits (B, T, vocab_size), list of cross_attn_weights per layer
        """
        B, T = captions.shape
        if tgt_mask is None:
            tgt_mask = self.generate_square_subsequent_mask(T)

        # Embed tokens and add positional encoding
        x = self.embedding(captions) * math.sqrt(self.d_model)  # (B,T,d)
        x = self.pos_encoding(x)

        # Optionally inject control signal as additive bias
        if control_signal is not None:
            ctrl = self.control_embed(control_signal).unsqueeze(1)  # (B,1,d)
            x = x + ctrl

        # Project encoder features
        memory = self.encoder_proj(encoder_features)  # (B, num_patches, d)

        all_attn = []
        for layer in self.layers:
            x, attn = layer(x, memory, tgt_mask=tgt_mask,
                            tgt_key_padding_mask=tgt_key_padding_mask)
            all_attn.append(attn)

        logits = self.output_proj(x)   # (B, T, vocab_size)
        return logits, all_attn


# ── Full Model ─────────────────────────────────────────────────────────
class ImageCaptioningModel(nn.Module):
    def __init__(self, vocab_size, encoder_dim=2048, d_model=512,
                 num_heads=8, num_layers=3, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.encoder = CNNEncoder(encoded_image_size=14)
        self.decoder = TransformerDecoder(
            vocab_size, d_model, num_heads, num_layers,
            dim_feedforward, encoder_dim, dropout
        )

    def forward(self, images, captions, tgt_key_padding_mask=None, control_signal=None):
        enc_feats = self.encoder(images)
        logits, attn = self.decoder(captions, enc_feats,
                                    tgt_key_padding_mask=tgt_key_padding_mask,
                                    control_signal=control_signal)
        return logits, attn

    @torch.no_grad()
    def generate(self, images, vocab, max_len=50, control_signal=None, beam_size=1):
        """
        Generate caption for a batch of images.
        beam_size=1 => greedy; beam_size>1 => beam search.
        Returns list of caption strings, list of attn weight lists.
        """
        self.eval()
        enc_feats = self.encoder(images)   # (B, num_patches, enc_dim)
        B = images.size(0)
        sos = vocab.word2idx[vocab.SOS]
        eos = vocab.word2idx[vocab.EOS]

        if beam_size == 1:
            return self._greedy_decode(enc_feats, vocab, sos, eos, max_len, control_signal)
        else:
            results = []
            for b in range(B):
                cap = self._beam_search(enc_feats[b:b+1], vocab, sos, eos,
                                        max_len, beam_size, control_signal)
                results.append(cap)
            return results, []

    def _greedy_decode(self, enc_feats, vocab, sos, eos, max_len, control_signal):
        B = enc_feats.size(0)
        tokens   = torch.full((B,1), sos, dtype=torch.long, device=enc_feats.device)
        finished = torch.zeros(B, dtype=torch.bool, device=enc_feats.device)
        all_attn = [[] for _ in range(B)]

        for _ in range(max_len - 1):
            logits, attn_list = self.decoder(tokens, enc_feats,
                                             control_signal=control_signal)
            next_tok = logits[:, -1, :].argmax(-1)  # (B,)
            # Save last layer cross-attention for last generated token
            last_attn = attn_list[-1][:, -1, :]     # (B, num_patches)
            for b in range(B):
                if not finished[b]:
                    all_attn[b].append(last_attn[b].cpu())

            finished = finished | (next_tok == eos)
            tokens   = torch.cat([tokens, next_tok.unsqueeze(1)], dim=1)
            if finished.all():
                break

        captions = [vocab.decode(tokens[b].tolist()) for b in range(B)]
        return captions, all_attn

    def _beam_search(self, enc_feat, vocab, sos, eos, max_len, beam_size, control_signal):
        """Single-image beam search."""
        # beam: list of (score, token_ids)
        beams     = [(0.0, [sos])]
        completed = []

        for _ in range(max_len):
            candidates = []
            for score, toks in beams:
                if toks[-1] == eos:
                    completed.append((score, toks))
                    continue
                inp    = torch.tensor([toks], dtype=torch.long, device=enc_feat.device)
                logits, _ = self.decoder(inp, enc_feat.expand(1, -1, -1),
                                          control_signal=control_signal)
                log_probs  = F.log_softmax(logits[0, -1], dim=-1)
                topk_vals, topk_ids = log_probs.topk(beam_size)
                for v, idx in zip(topk_vals.tolist(), topk_ids.tolist()):
                    candidates.append((score + v, toks + [idx]))
            if not candidates:
                break
            beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_size]

        if not completed:
            completed = beams
        best = max(completed, key=lambda x: x[0])
        return vocab.decode(best[1])


# ── Instantiate ────────────────────────────────────────────────────────
VOCAB_SIZE     = len(vocab)
D_MODEL        = 512
NUM_HEADS      = 8
NUM_LAYERS     = 3
DIM_FEEDFORWARD= 2048
DROPOUT        = 0.1

model = ImageCaptioningModel(
    vocab_size      = VOCAB_SIZE,
    d_model         = D_MODEL,
    num_heads       = NUM_HEADS,
    num_layers      = NUM_LAYERS,
    dim_feedforward = DIM_FEEDFORWARD,
    dropout         = DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {train_params:,}')

## **6. Defining Loss Function and Optimizer**


In [ ]:
PAD_IDX   = vocab.word2idx[vocab.PAD]
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)

# AdamW with different LR for encoder (fine-tuning) vs decoder
optimizer = torch.optim.AdamW([
    {'params': model.encoder.parameters(), 'lr': 1e-5},
    {'params': model.decoder.parameters(), 'lr': 1e-4},
], weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

print('Loss, optimizer and scheduler defined.')

## **7. Training the Image Captioning Model**


In [ ]:
CHECKPOINT_PATH = 'best_model.pth'


def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for images, captions, lengths, attn_mask in tqdm(dataloader, desc='Train', leave=False):
        images   = images.to(device)
        captions = captions.to(device)
        attn_mask= attn_mask.to(device)

        # Teacher forcing: input = captions[:,:-1], target = captions[:,1:]
        inp = captions[:, :-1]
        tgt = captions[:, 1:]
        pad_mask = attn_mask[:, :-1]

        logits, _ = model(images, inp, tgt_key_padding_mask=pad_mask)
        # logits: (B, T-1, V)
        B, T, V = logits.shape
        loss = criterion(logits.reshape(B*T, V), tgt.reshape(B*T))

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for images, captions, lengths, attn_mask in tqdm(dataloader, desc='Val', leave=False):
            images    = images.to(device)
            captions  = captions.to(device)
            attn_mask = attn_mask.to(device)
            inp = captions[:, :-1]
            tgt = captions[:, 1:]
            pad_mask = attn_mask[:, :-1]
            logits, _ = model(images, inp, tgt_key_padding_mask=pad_mask)
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B*T, V), tgt.reshape(B*T))
            total_loss += loss.item()
    return total_loss / len(dataloader)


def train(model, train_loader, val_loader, criterion, optimizer, scheduler,
          num_epochs, device, patience=5, checkpoint_path=CHECKPOINT_PATH):
    train_losses, val_losses = [], []
    best_val = float('inf')
    no_improve = 0

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        tr_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        vl_loss = validate(model, val_loader,   criterion, device)
        scheduler.step(vl_loss)

        train_losses.append(tr_loss)
        val_losses.append(vl_loss)
        elapsed = time.time() - t0
        print(f'Epoch {epoch:3d} | train_loss={tr_loss:.4f} | val_loss={vl_loss:.4f} | {elapsed:.0f}s')

        if vl_loss < best_val:
            best_val = vl_loss
            no_improve = 0
            torch.save({'epoch': epoch,
                        'model_state': model.state_dict(),
                        'optimizer_state': optimizer.state_dict(),
                        'val_loss': vl_loss}, checkpoint_path)
            print(f'  ✓ Checkpoint saved (val_loss={vl_loss:.4f})')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping after {epoch} epochs.')
                break

    return train_losses, val_losses


NUM_EPOCHS = 10
PATIENCE   = 4

train_losses, val_losses = train(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    num_epochs=NUM_EPOCHS, device=device, patience=PATIENCE
)

## **8.1 Visualizing Training Metrics**


In [ ]:
# Restore best checkpoint
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state'])
print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f})")


def plot_losses(train_losses, val_losses):
    epochs = range(1, len(train_losses)+1)
    plt.figure(figsize=(9,4))
    plt.plot(epochs, train_losses, 'o-', label='Train Loss')
    plt.plot(epochs, val_losses,   's-', label='Val Loss')
    plt.xlabel('Epoch'); plt.ylabel('Cross-Entropy Loss')
    plt.title('Training and Validation Loss')
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()


plot_losses(train_losses, val_losses)

## **8.2 Visualizing Cross-Attention Weights**


In [ ]:
GRID_SIZE = 14   # encoder outputs 14x14 patches


def visualize_cross_attention(image_tensor, caption, attention_per_word, max_words=10):
    """
    image_tensor:      (3,H,W) normalized tensor
    caption:           string
    attention_per_word: list of (num_patches,) tensors, one per generated token
    """
    words   = caption.split()[:max_words]
    n_words = min(len(words), len(attention_per_word), max_words)
    img_np  = denormalize(image_tensor).permute(1,2,0).numpy()

    cols = min(n_words + 1, 6)
    rows = math.ceil((n_words + 1) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.5*cols, 3.5*rows))
    axes = np.array(axes).flatten()

    # Original image
    axes[0].imshow(img_np); axes[0].set_title('Original'); axes[0].axis('off')

    for i in range(n_words):
        attn = attention_per_word[i].detach().cpu().float()
        # Reshape to grid and upsample
        attn_map = attn[:GRID_SIZE*GRID_SIZE].reshape(GRID_SIZE, GRID_SIZE).numpy()
        attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)
        h, w = img_np.shape[:2]
        attn_resized = np.array(Image.fromarray((attn_map*255).astype(np.uint8)).resize((w,h),
                       Image.BILINEAR)) / 255.0

        axes[i+1].imshow(img_np)
        axes[i+1].imshow(attn_resized, alpha=0.55, cmap='jet')
        axes[i+1].set_title(f'"{words[i]}"', fontsize=10)
        axes[i+1].axis('off')

    for ax in axes[n_words+1:]:
        ax.axis('off')
    plt.tight_layout(); plt.show()


# Show attention maps for 5 validation images
model.eval()
shown = 0
for images, captions, lengths, attn_mask in val_loader:
    imgs = images[:1].to(device)
    caps, attn_per_img = model.generate(imgs, vocab, max_len=20)
    if attn_per_img[0]:   # has attention
        visualize_cross_attention(images[0], caps[0], attn_per_img[0])
        shown += 1
    if shown >= 5:
        break

## **8.3 Running Inference on the Image Captioning Model**


In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction


def length_to_control(length: int) -> int:
    """Map target length to control bucket index."""
    if length < 10:   return 0
    if length < 20:   return 1
    if length < 35:   return 2
    return 3


@torch.no_grad()
def generate_caption(model, image_tensor, vocab, max_len=50,
                     control_signal=None, beam_size=1):
    """
    image_tensor: (3,H,W) or (1,3,H,W)
    Returns: caption string, list of attention tensors per word
    """
    model.eval()
    if image_tensor.dim() == 3:
        image_tensor = image_tensor.unsqueeze(0)
    image_tensor = image_tensor.to(device)
    ctrl = None
    if control_signal is not None:
        ctrl = torch.tensor([control_signal], device=device)
    captions, attn_list = model.generate(image_tensor, vocab,
                                          max_len=max_len,
                                          control_signal=ctrl,
                                          beam_size=beam_size)
    return captions[0], attn_list[0]


@torch.no_grad()
def evaluate_model(model, dataloader, vocab, max_samples=1000):
    model.eval()
    smooth = SmoothingFunction().method1
    refs_corpus, hyps_corpus = [], []
    count = 0

    for images, captions, lengths, _ in tqdm(dataloader, desc='Evaluating'):
        imgs = images.to(device)
        gen_caps, _ = model.generate(imgs, vocab, max_len=50)
        for b, cap in enumerate(gen_caps):
            hyp = cap.split()
            # use ground-truth from the batch (one ref per sample here)
            ref = [vocab.decode(captions[b].tolist()).split()]
            refs_corpus.append(ref)
            hyps_corpus.append(hyp)
        count += len(gen_caps)
        if count >= max_samples:
            break

    scores = {
        'BLEU-1': corpus_bleu(refs_corpus, hyps_corpus, weights=(1,0,0,0)),
        'BLEU-2': corpus_bleu(refs_corpus, hyps_corpus, weights=(.5,.5,0,0)),
        'BLEU-3': corpus_bleu(refs_corpus, hyps_corpus, weights=(.33,.33,.33,0)),
        'BLEU-4': corpus_bleu(refs_corpus, hyps_corpus, weights=(.25,.25,.25,.25)),
    }
    return scores


# ── Evaluate ───────────────────────────────────────────────────────────
custom_scores = evaluate_model(model, val_loader, vocab, max_samples=1000)
print('\nCustom Model BLEU Scores:')
for k, v in custom_scores.items():
    print(f'  {k}: {v:.4f}')

# ── Show 10 sample captions ─────────────────────────────────────────────
sample_images, sample_caps, sample_lens, _ = next(iter(val_loader))
model.eval()
gen_captions, _ = model.generate(sample_images[:10].to(device), vocab, max_len=40)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, ax in enumerate(axes.flatten()):
    img = denormalize(sample_images[i]).permute(1,2,0).numpy()
    gt  = vocab.decode(sample_caps[i].tolist())
    ax.imshow(img); ax.axis('off')
    ax.set_title(f'GT: {gt[:60]}\nGen: {gen_captions[i][:60]}', fontsize=7)
plt.suptitle('Ground-truth vs Generated Captions', fontsize=12)
plt.tight_layout(); plt.show()

# ── Controllable generation demo ────────────────────────────────────────
print('\n=== Controllable Generation Demo ===')
demo_img = sample_images[0]
for ctrl, label in [(0,'short'), (1,'medium'), (2,'long'), (3,'very long')]:
    cap, _ = generate_caption(model, demo_img, vocab, max_len=50, control_signal=ctrl)
    print(f'  [{label}] {cap}')

## **9. Loading an Existing Image Captioning Model**


In [ ]:
# !pip install transformers accelerate
from transformers import BlipProcessor, BlipForConditionalGeneration


def load_pretrained_model(model_id='Salesforce/blip-image-captioning-base'):
    """Load BLIP pre-trained model and processor."""
    processor = BlipProcessor.from_pretrained(model_id)
    blip      = BlipForConditionalGeneration.from_pretrained(model_id).to(device)
    blip.eval()
    print(f'Loaded pre-trained model: {model_id}')
    return blip, processor


pretrained_model, blip_processor = load_pretrained_model()

## **10. Evaluating the Existing Image Captioning Model**


In [ ]:
@torch.no_grad()
def evaluate_pretrained_model(model, processor, dataloader,
                               vocab, max_samples=1000):
    """
    Evaluate BLIP on the same val images using the same BLEU metric.
    We denormalize tensors back to PIL to feed into the BLIP processor.
    """
    model.eval()
    smooth = SmoothingFunction().method1
    refs_corpus, hyps_corpus = [], []
    count = 0

    for images, captions, lengths, _ in tqdm(dataloader, desc='Evaluating BLIP'):
        # Convert tensor batch back to PIL
        pil_images = []
        for img in images:
            pil = transforms.ToPILImage()(denormalize(img))
            pil_images.append(pil)

        inputs = processor(images=pil_images, return_tensors='pt').to(device)
        out    = model.generate(**inputs, max_new_tokens=50)
        gen    = processor.batch_decode(out, skip_special_tokens=True)

        for b, cap in enumerate(gen):
            hyp = cap.lower().split()
            ref = [vocab.decode(captions[b].tolist()).split()]
            refs_corpus.append(ref)
            hyps_corpus.append(hyp)
        count += len(gen)
        if count >= max_samples:
            break

    scores = {
        'BLEU-1': corpus_bleu(refs_corpus, hyps_corpus, weights=(1,0,0,0)),
        'BLEU-2': corpus_bleu(refs_corpus, hyps_corpus, weights=(.5,.5,0,0)),
        'BLEU-3': corpus_bleu(refs_corpus, hyps_corpus, weights=(.33,.33,.33,0)),
        'BLEU-4': corpus_bleu(refs_corpus, hyps_corpus, weights=(.25,.25,.25,.25)),
    }
    return scores


pretrained_scores = evaluate_pretrained_model(
    pretrained_model, blip_processor, val_loader, vocab, max_samples=1000
)
print('\nBLIP Pre-trained Model BLEU Scores:')
for k, v in pretrained_scores.items():
    print(f'  {k}: {v:.4f}')

## **11. Comparing the Two Models**


In [ ]:
def compare_models(custom_results, pretrained_results):
    metrics = list(custom_results.keys())
    custom_vals     = [custom_results[m]     for m in metrics]
    pretrained_vals = [pretrained_results[m] for m in metrics]

    # Table
    print(f"{'Metric':<10} {'Custom':>10} {'BLIP':>10} {'Delta':>10}")
    print('-' * 45)
    for m, cv, pv in zip(metrics, custom_vals, pretrained_vals):
        print(f'{m:<10} {cv:>10.4f} {pv:>10.4f} {pv-cv:>+10.4f}')

    # Bar chart
    x = np.arange(len(metrics))
    width = 0.35
    fig, ax = plt.subplots(figsize=(9,5))
    ax.bar(x - width/2, custom_vals,     width, label='Custom (ResNet+Transformer)')
    ax.bar(x + width/2, pretrained_vals, width, label='BLIP (pre-trained)')
    ax.set_xticks(x); ax.set_xticklabels(metrics)
    ax.set_ylabel('BLEU Score'); ax.set_title('BLEU Score Comparison')
    ax.legend(); ax.grid(axis='y', alpha=0.4)
    plt.tight_layout(); plt.show()


compare_models(custom_scores, pretrained_scores)

# ── Side-by-side caption comparison for 10 images ──────────────────────
sample_images, sample_caps, _, _ = next(iter(val_loader))
pil_imgs   = [transforms.ToPILImage()(denormalize(img)) for img in sample_images[:10]]
blip_inputs = blip_processor(images=pil_imgs, return_tensors='pt').to(device)
blip_out    = pretrained_model.generate(**blip_inputs, max_new_tokens=50)
blip_caps   = blip_processor.batch_decode(blip_out, skip_special_tokens=True)

model.eval()
custom_caps, _ = model.generate(sample_images[:10].to(device), vocab, max_len=40)

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
for i, ax in enumerate(axes.flatten()):
    img = denormalize(sample_images[i]).permute(1,2,0).numpy()
    gt  = vocab.decode(sample_caps[i].tolist())
    ax.imshow(img); ax.axis('off')
    ax.set_title(
        f'GT:     {gt[:55]}\n'
        f'Custom: {custom_caps[i][:55]}\n'
        f'BLIP:   {blip_caps[i][:55]}',
        fontsize=6.5
    )
plt.suptitle('Caption Comparison: Custom vs BLIP', fontsize=13)
plt.tight_layout(); plt.show()

> **Answer:**
>
> **1. Why does the pre-trained model perform better/worse on certain images?**
> BLIP was trained on hundreds of millions of image-text pairs using a unified vision-language pre-training objective (contrastive + captioning + matching). This gives it much stronger generalisation, especially for rare objects, complex scenes, and fine-grained attributes. Our custom model only sees ≤20k COCO training images, so it struggles with uncommon vocabulary and unusual compositions. BLIP performs *worse* on very specific COCO annotation styles (e.g. counting, spatial relations) because its captions tend to be more natural but less detailed.
>
> **2. What architectural differences contribute to performance gaps?**
> - **Encoder:** BLIP uses a Vision Transformer (ViT-B/16) which captures global context from the first layer, whereas our ResNet-50 CNN captures local features hierarchically. ViT patch embeddings integrate better with the Transformer decoder.
> - **Pre-training scale:** BLIP was jointly pre-trained end-to-end on web-scale data; our encoder was only fine-tuned from ImageNet weights on a small COCO subset.
> - **Decoder capacity:** BLIP's decoder is a full BERT/GPT-style pre-trained language model; ours is a 3-layer Transformer trained from scratch.
>
> **3. How does your Transformer decoder compare to BLIP's decoder?**
> Our decoder is a lightweight 3-layer Transformer (512d, 8 heads) trained from scratch on ~20k images. BLIP's decoder is a pre-trained 12-layer Transformer with 768d, already possessing rich language priors. The main bottleneck for our model is not the decoder architecture per se, but the limited training data and the weaker visual encoder.
>
> **4. Three specific improvements for the custom model:**
> 1. **Replace ResNet with a Vision Transformer (ViT-B/16)** – patch-level features integrate more naturally with the Transformer decoder and yield richer spatial representations.
> 2. **Increase training data / use data augmentation more aggressively** – use the full 118k COCO training set and apply RandAugment/CutMix to improve robustness.
> 3. **Adopt a pre-trained language model as the decoder** (e.g. GPT-2 or a small BERT variant) via weight initialisation, so the model starts with strong language priors instead of training the decoder from a random initialisation.

---
**Remember to save and upload your `.ipynb` file to Canvas before the deadline!**